# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Session 7: TLS 1.3

**45 minutes taught · 75–90 minutes independently.** Instructor (see course website) · Notebook (see course website)

## Outcomes and setup

Explain how TLS combines certificate authentication, transcript authentication, key establishment and record protection. Identify where plaintext exists when a connection terminates. Distinguish server-authenticated TLS, mTLS and application authorization. Complete Session 6 and use the Day 2 setup (see course website); helpers are embedded in the notebook and demo.



## The protocol puts the pieces together

TLS is a specified protocol, not simply AES applied to a socket. In a typical full certificate-based TLS 1.3 handshake, peers exchange public key-establishment contributions, authenticate the handshake, and derive keys. CertificateVerify proves control of the signing key over specified handshake context; Finished authenticates the transcript using derived key material. A certificate alone does not prove the current peer possesses its private key.

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    C->>S: ClientHello with offered parameters and key share
    S->>C: ServerHello with selection and key share
    Note over C,S: Derive handshake protection keys
    S->>C: EncryptedExtensions and certificate authentication
    S->>C: Finished
    C->>C: Validate identity, signature and Finished
    C->>S: Finished and optional requested client authentication
    Note over C,S: Application records use derived traffic keys
```

This is a conceptual full-handshake sequence, not a packet parser. In mTLS, the requested client Certificate and CertificateVerify precede the client's Finished. Resumption and early data change the flow. Follow the protocol rather than copying this sketch into a new handshake implementation.

TLS 1.3 cipher-suite names identify record protection and a hash. The selected key-establishment group and signature scheme are separate. Seeing `TLS_AES_256_GCM_SHA384` does not prove ML-KEM was used, nor identify the certificate's signature algorithm.

## Observe a real in-memory handshake


In [ ]:
result = tls_trial()
assert result['version'] == 'TLSv1.3'
assert result['application_bytes'] > 0
assert not result['client_authenticated']
print(result)
print('PASS: TLS 1.3 carried application bytes with server authentication')


The helper connects two real SSL objects through MemoryBIO buffers. It does not simulate the cryptographic TLS implementation, open a listening port, or contact an internet server. It creates a trusted test CA in the client context only. Python's `ssl` backend can differ from the backend bundled with `cryptography`; supporting ML-KEM in one does not establish PQ support in the other.

## Where confidentiality ends

```mermaid
flowchart LR
    C["Client plaintext"] --> T["TLS connection one"]
    T --> P["Proxy terminates TLS: plaintext available here"]
    P --> U["TLS connection two"]
    U --> B["Backend plaintext"]
    P --> L["Logs, tracing and administrators may see content"]
```

Read the two connections separately. Re-encryption protects the proxy-to-backend path; it does not hide content from the proxy. A security requirement for end-to-end confidentiality must identify the actual endpoints and who may inspect plaintext. TLS also does not hide all traffic metadata or protect application data after decryption.

| Requirement | TLS contribution | Remaining application work |
| --- | --- | --- |
| Prevent network modification | Authenticated records within the connection | Validate data and actions after decryption |
| Identify intended server | Validated certificate and handshake authentication | Supply correct expected identity and trust policy |
| Identify client workload | mTLS when configured and validated | Map identity to authorized operations |
| Keep records confidential for decades | Protect current transport under chosen assumptions | Assess retained ciphertext, endpoint storage and PQ migration |

## Failures are security results


In [ ]:
expect_rejection(lambda: tls_trial(hostname='attacker.test'), ssl.SSLError)
expect_rejection(lambda: tls_trial(trust_root=False), ssl.SSLError)
assert tls_trial(mtls=True)['client_authenticated']
expect_rejection(lambda: tls_trial(mtls=True, send_client=False), ssl.SSLError)
print('PASS: name, trust and required client authentication are enforced')


Do not fix these failures by disabling hostname or certificate verification. Fix issuance, chain delivery, expected identity or trust configuration. Missing client authentication must not silently fall back to anonymous privileges.

## Forward secrecy, resumption and early data

Fresh ephemeral DH in an authenticated handshake can protect past traffic against later theft of a long-term authentication key, assuming ephemeral and traffic secrets were erased. This is not a promise against endpoint compromise or a future solver of the classical key-exchange problem.

Resumption uses PSKs derived from earlier sessions; its security properties depend on how the PSK and any new exchange are combined. TLS 1.3 0-RTT early data has replay risks and weaker forward-secrecy properties than normal application data. It should not be treated as ordinary replay-protected delivery of payments, approvals or other non-idempotent operations. Our examples intentionally use full handshakes and no early data.

```mermaid
flowchart TD
    R["Proposed TLS deployment"] --> E["Locate every termination endpoint"]
    E --> I["Specify trusted identities on each hop"]
    I --> A["Map authenticated clients to permissions"]
    A --> S["Review resumption, early data and secret retention"]
    S --> M["Measure negotiated groups and monitor failures"]
```

For a powerful adversary, protecting the private key alone is insufficient if a load balancer, debug logger or deployment administrator can access plaintext. Draw those boundaries before selecting a cipher suite. Use Lab 4 (see course website) to make successful and failed trust decisions observable.

## Practice and answers

1. Does a TLS AES-256 cipher name establish post-quantum key exchange?
2. A proxy decrypts then re-encrypts. Is content hidden from that proxy?
3. Does a valid mTLS certificate authorize deleting all invoices?
4. Why should an invoice-approval request not be assumed replay-safe in 0-RTT?

<details><summary>Worked answers</summary>
<ol><li>No. Inspect the negotiated group and authentication mechanisms separately.</li><li>No. It is an endpoint of both connections.</li><li>No. The authenticated identity needs application authorization.</li><li>Early data can be replayed; the application needs an appropriate policy and duplicate handling.</li></ol>
</details>

## Sources

Reviewed 22 September 2026: [RFC 8446](https://www.rfc-editor.org/rfc/rfc8446) and [Python SSLObject/MemoryBIO](https://docs.python.org/3/library/ssl.html). This session does not claim that its TLS connection negotiates PQ or hybrid groups.


In [ ]:
print("PASS: completed session-07-tls demonstrations; learner status is reported separately")
